# TensorFlow Model (2.17.1)

[Model Explorer](https://github.com/google-ai-edge/model-explorer?tab=readme-ov-file) access to the TensorFlow Model.

[Original Repository](https://github.com/googlecodelabs/firebase-iap-optimization)

[Use-case Guide](https://firebase.google.com/codelabs/iap-optimization?hl=en#6)

TensorFlow guides used:
- [Customizing what happens in Model.fit](https://www.tensorflow.org/guide/keras/customizing_what_happens_in_fit)
- [Working with preprocessing layers](https://www.tensorflow.org/guide/keras/preprocessing_layers)
- **[Sample Weights](https://www.tensorflow.org/guide/keras/training_with_built_in_methods#using_sample_weighting_and_class_weighting)
- **[Personalized Loss Function](https://www.tensorflow.org/guide/keras/training_with_built_in_methods#handling_losses_and_metrics_that_dont_fit_the_standard_signature)
- **[Using Callbacks](https://www.tensorflow.org/guide/keras/training_with_built_in_methods#using_callbacks)
- **[Adding Loss inside custom training step](https://www.tensorflow.org/guide/keras/making_new_layers_and_models_via_subclassing#the_add_loss_method)
- [Fault-tolerant training](https://www.tensorflow.org/guide/keras/training_with_built_in_methods#checkpointing_models)
- [Reduce learning rate as train progresses](https://www.tensorflow.org/guide/keras/training_with_built_in_methods#using_learning_rate_schedules)

Extracting features from layers: [Extracting features from layers in TensorFlow](https://www.tensorflow.org/guide/keras/sequential_model#feature_extraction_with_a_sequential_model)

In [1]:
"""Data prep, train and evaluate DNN model."""

import logging
import os
from datetime import datetime
import json

import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Concatenate, Reshape
from tflite_support import flatbuffers
from tflite_support import metadata as _metadata
from tflite_support import metadata_schema_py_generated as _metadata_fb

logging.info(tf.version.VERSION)
logging.basicConfig(level=logging.INFO)

# Config Params

In [ ]:
DATE = datetime.now().strftime("%Y%m%d")

# Used by CSV reader
COLUMNS = {
    "distance_avg": {"default": [0.0], "type": tf.float32},
    "coins_spent": {"default": [0.0], "type": tf.float32},
    "game_day": {"default": [0.0], "type": tf.float32},
    "geo_country": {"default": ["na"], "type": tf.string},
    "device_os": {"default": ["na"], "type": tf.string},
    "last_run_end_reason": {"default": ["na"], "type": tf.string},
    "presented_powerup": {"default": ["na"], "type": tf.string},
    "is_powerup_clicked": {
        "default": [0],
        "type": tf.int32,
    },  # Preprocess must enforce this to be a 1 or 0
    "event_timestamp": {"default": ["na"], "type": tf.string},
    "event_id": {"default": ["na"], "type": tf.string},
}
CSV_COLUMNS = list(COLUMNS.keys())
DEFAULTS = [COLUMNS[col]["default"] for col in CSV_COLUMNS]
SHUFFLE_BUFFER = 1000
SAMPLED = True

DATA_FILE_PATH = "/Users/Roberto_Aguilar/PycharmProjects/adaptive_contextual_bandits_dnn/data/raw/"
if SAMPLED:
    FILE_PATH = DATA_FILE_PATH + "training.csv"
else:
    FILE_PATH = DATA_FILE_PATH + "training.csv"

# Used by loader
LABEL_COLUMN = "is_powerup_clicked" # reward label
UNWANTED_COLS = ["event_timestamp", "event_id"]
ACTION_WEIGHT_COLUMN = "presented_powerup"

# Model Training
EPOCHS = 3  # Default: 10
STEPS_PER_EPOCH = 1000  # Default: 256

# Learning Rate Decay
LR_DECAY_STEPS = 1500  # Default: 1000
LR_DECAY_RATE = 0.90  # Default: 0.96
LR_STAIRCASE = True  # Default: True

# Optimizer
INITIAL_LEARNING_RATE = 0.0002  # Default: 0.001

# Used by transform layer
INPUT_COLS = {
    colname: colinfo
    for colname, colinfo in COLUMNS.items()
    if colname is not LABEL_COLUMN and colname not in UNWANTED_COLS
}

# Model Saving (also checkpoint saving)
MODEL_DIR = f"models/{DATE}/"
KERAS_DIR = MODEL_DIR + "keras/"
H5_DIR = MODEL_DIR + "h5/"
TFLITE_DIR = MODEL_DIR + "tflite/"

SAVE_FORMAT = "tf"  # or "h5" for SavedModel
if not os.path.exists(MODEL_DIR):
    os.makedirs(KERAS_DIR)
    os.makedirs(H5_DIR)
    os.makedirs(TFLITE_DIR)

print(f"CSV_COLUMNS: {CSV_COLUMNS}\n")
print(f"LABEL_COLUMN: {LABEL_COLUMN}\n")
print(f"UNWANTED_COLS: {UNWANTED_COLS}\n")
print(f"INPUT_COLS: {list(INPUT_COLS.keys())}")

CSV_COLUMNS: ['distance_avg', 'coins_spent', 'game_day', 'geo_country', 'device_os', 'last_run_end_reason', 'presented_powerup', 'is_powerup_clicked', 'event_timestamp', 'event_id']

LABEL_COLUMN: is_powerup_clicked

UNWANTED_COLS: ['event_timestamp', 'event_id']

INPUT_COLS: ['distance_avg', 'coins_spent', 'game_day', 'geo_country', 'device_os', 'last_run_end_reason', 'presented_powerup']


Data Original Columns:

| ID | Column               | Non-Null Count | Dtype    |
|----|----------------------|----------------|----------|
| 0  | distance_avg         | 522258         | int64    |
| 1  | coins_spent          | 522258         | float64  |
| 2  | game_day             | 522258         | float64  |
| 3  | geo_country          | 522258         | object   |
| 4  | device_os            | 522258         | object   |
| 5  | last_run_end_reason  | 522258         | object   |
| 6  | presented_powerup    | 522258         | object   |
| 7  | is_powerup_clicked   | 522258         | int64    |
| 8  | event_timestamp      | 522258         | object   |
| 9  | event_id             | 522258         | object   |

In [8]:
data = pd.read_csv(FILE_PATH)
data.head()

,event_id,event_timestamp,distance_avg,coins_spent,game_day,geo_country,device_os,last_run_end_reason,presented_powerup,is_powerup_clicked
0,0,2025-10-18 21:17:16,102,439,13,Canada,Android,laser,coin_magnet,0
1,1,2025-07-28 15:21:50,85,1296,170,Japan,Android,wall,parachute,0
2,2,2025-10-20 08:38:02,134,956,5,US,iOS,wall,time_machine,0
3,3,2025-11-30 05:23:19,87,661,8,UK,iOS,laser,time_machine,0
4,4,2024-09-01 00:03:08,175,0,65,US,Android,wall,extra_life,0


# Data Preprocessing

## Layer 1: FillNA

Layer to fill missing values in the dataset. This is going to be used to fill missing values in the dataset.

In [9]:
class FillNA(tf.keras.layers.Layer):
    def __init__(self, fill_values=None, name=None, **kwargs):
        """
        Initialize the FillNA layer.
        Args:
            fill_values (dict): A dictionary mapping data types to fill values.
            Example: {tf.float32: 0.0, tf.int32: -1, tf.string: ""}
        """
        super(FillNA, self).__init__(name=name, **kwargs)
        self.fill_values = fill_values or {
            tf.float32: 0.0,
            tf.int32: -1,
            tf.int64: -1,
            tf.string: "N/A",
        }

    def call(self, inputs):
        """
        Perform the forward pass to fill missing values.
        Args:
            inputs (tf.Tensor): Input tensor with potential missing values.
        Returns:
            tf.Tensor: Tensor with missing values filled.
        """
        dtype = inputs.dtype
        fill_value = self.fill_values.get(dtype, 0)

        if dtype == tf.float32:
            return tf.where(tf.math.is_nan(inputs), fill_value, inputs)
        elif dtype.is_integer:
            return tf.where(tf.equal(inputs, 0), fill_value, inputs)
        elif dtype == tf.string:
            return tf.where(tf.equal(inputs, ""), fill_value, inputs)
        else:
            raise ValueError(f"Unsupported data type: {dtype}")

    def get_config(self):
        """
        Return the configuration of the layer for serialization.
        """
        config = super(FillNA, self).get_config()
        config.update({"fill_values": self.fill_values})
        return config

## Sample Weighting

### 1. Precompute the action weights for `create_dataset`

First we need to compute the weights for the actions. This is done by calculating the frequency of each action in the dataset.

In [10]:
def calc_action_space(dataset: tf.data.Dataset) -> dict:
    """
    Calculate the action space, actions, and actions mapping.
    Example:
        Action Mapping
        >> {0: b"coin_magnet", 1: b"coin_multiplier", ...}
    """
    actions = dataset.map(
        lambda row_data: row_data[ACTION_WEIGHT_COLUMN], name="actions_map"
    )
    actions_np = np.concatenate(list(actions.as_numpy_iterator()))

    unique_actions, action_counts = np.unique(actions_np, return_counts=True)
    actions_mapping = {i: action for i, action in enumerate(unique_actions)}
    action_space_size = len(unique_actions)

    return {
        "actions": actions,
        "actions_np": actions_np,
        "unique_actions": unique_actions,
        "action_counts": action_counts,
        "actions_mapping": actions_mapping,
        "action_space": unique_actions,
        "action_space_size": action_space_size,
    }


def prep_actions_weights(dataset: tf.data.Dataset) -> dict:
    """
    Prepare the weights for the actions, this calculates the sample weight based on
    frequency of reward showing up.
    Args:
        dataset (tf.data.Dataset): The dataset containing the actions.
    Returns:
        dict: A dictionary mapping action values to weights.
    Note:
        Sample with reward of 1 is 2x the weight of those with reward of 0. This is
        decided based on the distribution of rewards. Then the samples with reward of
         1 is scaled based on their distribution.
    Example:
        Normalized Weights
        >> {b'coin_magnet': 0.12456487023654975, b'coin_multiplier': 0.12500909512156827,
            b'extra_life': 0.12516036135396683, b'head_start': 0.124164684887546,
            b'nuclear_missle': 0.12570606864806283, b'parachute': 0.12506462323219558,
            b'sparky_armor': 0.1244136040041512, b'time_machine': 0.12591669251595955}

        Actions Weights
        >> {b'coin_magnet': 4.00698030554317, b'coin_multiplier': 3.999854485995665,
            b'extra_life': 3.997436684413578, b'head_start': 4.0134324011409745,
            b'nuclear_missle': 3.9887505366216107, b'parachute': 3.9989664290237927,
            b'sparky_armor': 4.009415476099381, b'time_machine': 3.9854131010989815}
    """
    logging.info("Preparing actions weights...")
    calc_dict = calc_action_space(dataset)

    normalized_counts = calc_dict["action_counts"] / calc_dict["action_counts"].sum()
    value_counts_normalized = dict(zip(calc_dict["unique_actions"], normalized_counts))

    return {key: (2 / value) ** 0.5 for key, value in value_counts_normalized.items()}

### 2. Create a sample weighting tensor for `features_and_labels`

Then we create a sample weighting tensor based on the action based on a weight. The sample weight is 1.0 if the label is 0, else the action weight is looked up by action key.

In [ ]:
def compute_sample_weight(
    presented_powerup: tf.Tensor, label: tf.Tensor, actions_weight: dict
) -> tf.Tensor:
    """
    Create a sample weight tensor based on the presented powerup (action) based on a
    weight.
    E.g.: Sample weight = 1.0 if label==0, else actions_weight looked up by action key.
    Args:
        presented_powerup (tf.Tensor): Tensor of presented_powerup values.
        label (tf.Tensor): Tensor of reward values.
        actions_weight (dict): Mapping of actions to weights.
    Returns:
        tf.Tensor: Computed sample weights.
    """
    logging.info(f"Computing Actions Weight: {actions_weight}")

    keys = tf.constant(list(actions_weight.keys()), dtype=tf.string)
    values = tf.constant(list(actions_weight.values()), dtype=tf.float32)
    weight_table = tf.lookup.StaticHashTable(
        initializer=tf.lookup.KeyValueTensorInitializer(keys, values),
        default_value=1.0,
    )

    fill_na_layer = FillNA(name="fill_na")
    presented_powerup = fill_na_layer(tf.cast(presented_powerup, tf.string))
    action_weight = weight_table.lookup(presented_powerup)

    # if label == 0 => weight=1.0, else weight=action_weight
    label_int = tf.cast(label, tf.int32)
    weight = tf.where(
        tf.equal(label_int, 0),
        tf.constant(1.0, dtype=tf.float32),
        action_weight,
    )
    return weight

## Data Ingestion

Create a dataset from the CSV file and transformed into a `tf.Dataset` object. The dataset is then split into training and evaluation datasets.

In [ ]:
def features_and_labels(row_data: dict, actions_weight: dict) -> tuple:
    """
    Extract features, label, and sample_weight from row_data.
    Args:
        row_data (dict): Row-level data from the dataset.
        actions_weight (dict): Mapping of the weight for each action type.
    Returns:
        tuple: A tuple of features, label, and sample_weight as tensors.
    """
    for unwanted_col in UNWANTED_COLS:
        row_data.pop(unwanted_col, None)

    label = tf.cast(row_data.pop(LABEL_COLUMN, 0.0), tf.float32)
    presented_powerup = row_data.get(ACTION_WEIGHT_COLUMN, "")

    return (
        row_data,
        label,
        compute_sample_weight(presented_powerup, label, actions_weight),
    )


def load_dataset(pattern: str, batch_size: int, num_repeat: int = 1):
    """
    Load and preprocess the dataset from CSV files.
    Args:
        pattern (str): File pattern for the dataset.
        batch_size (int): Batch size for dataset.
        num_repeat (int): Number of epochs for dataset iteration.
    Returns:
        tf.data.Dataset: Preprocessed dataset.
    """
    try:
        dataset = tf.data.experimental.make_csv_dataset(
            file_pattern=pattern,
            batch_size=batch_size,
            column_names=CSV_COLUMNS,
            column_defaults=DEFAULTS,
            num_epochs=num_repeat,  # Set to None for infinite iterations
            shuffle_buffer_size=SHUFFLE_BUFFER,
            field_delim="|",
            compression_type="GZIP",
        )
        return dataset
    except Exception as e:
        logging.error(f"Error loading dataset from {pattern}: {e}")
        raise


def create_dataset(pattern: str, batch_size: int) -> tf.data.Dataset:
    """Create dataset object.
    Note:
        Action Weights are pre-computed from the full dataset distribution once to
        create the set of features.
    """
    dataset = load_dataset(pattern, batch_size)
    actions_weight = prep_actions_weights(dataset)
    dataset = dataset.map(
        lambda row_data: features_and_labels(row_data, actions_weight)
    )
    return dataset.prefetch(tf.data.AUTOTUNE)


def split_dataset(dataset: tf.data.Dataset, train_ratio: float = 0.8) -> tuple:
    """
    Split dataset into train and eval datasets.
    """
    dataset = dataset.enumerate()

    dataset_size = sum(1 for _ in dataset)
    if dataset_size == 0:
        raise ValueError("Dataset is empty and cannot be split.")

    train_size = int(train_ratio * dataset_size)
    if train_size == 0 or train_size == dataset_size:
        raise ValueError(
            f"Invalid train_ratio: {train_ratio}. Train size: {train_size},"
            f" Eval size: {dataset_size - train_size}"
        )

    # Filter and map back to original data format
    train_ds = (
        dataset.filter(lambda idx, _: idx < train_size)
        .map(lambda _, data: data)
        .prefetch(tf.data.AUTOTUNE)
    )
    eval_ds = (
        dataset.filter(lambda idx, _: idx >= train_size)
        .map(lambda _, data: data)
        .prefetch(tf.data.AUTOTUNE)
    )

    return train_ds, eval_ds

## Create the dataset

In [ ]:
logging.info("Creating dataset...")
dataset = create_dataset(pattern=FILE_PATH, batch_size=32)

In [ ]:
logging.info("Splitting dataset...")
train_dataset, eval_dataset = split_dataset(dataset, train_ratio=0.8)

# # Verify the output
# for features, labels, sample_weights in train_dataset.take(1):
# print("Features:", features)
# print("Labels:", labels)
# print("Sample Weights:", sample_weights)

# Layer Creation Helpers

## Layer 2: Category Encoding (One-Hot Encoding)

In [ ]:
def create_one_hot_encoding_layer(
    name: str, dataset: tf.data.Dataset, dtype: str = "int", max_tokens: int = None
) -> tf.keras.layers.Layer:
    """
    Create a one-hot encoding layer for a categorical feature.
    Args:
        name (str): The feature name to be encoded (key in the features dict).
        dataset (tf.data.Dataset): The dataset yielding triplets (features, label, sample_weight).
        dtype (str): Either "string" or "int"/"integer" indicating the feature data type.
        max_tokens (int): The maximum number of unique values for the lookup.
    Returns:
        tf.keras.layers.Layer: Return a small callable (Lmabda) that takes a tensor
        of feature values, passes them through 'index' -> 'encoder', returning one-hot vectors.
    Note:
        This approach can be more efficient than just using Lookup, in scenarios
        where you need to handle multiple encoding schemes or large datasets.
    """
    # maps string/integer to an integer ID
    if dtype.lower() == "string":
        index = tf.keras.layers.StringLookup(
            max_tokens=max_tokens,
            output_mode="int",
            name=name + "_string_lookup",
        )
    elif "int" in dtype.lower():
        index = tf.keras.layers.IntegerLookup(
            max_tokens=max_tokens,
            output_mode="int",
            name=name + "_integer_lookup",
        )
    else:
        raise ValueError(f"Unsupported data type: {dtype}. Must be 'string' or 'int'.")

    feature_ds = dataset.map(
        lambda features, label, sample_weight: features[name],
        name=f"{name}_encoder_map",
    )
    # "Adapt" the index layer to learn the unique tokens from the feature. This is a
    # one-time & offline step that builds the vocabulary
    index.adapt(feature_ds)

    encoder = tf.keras.layers.CategoryEncoding(
        num_tokens=index.vocabulary_size(),
        output_mode="one_hot",
        name=name + "_category_encoder",
    )
    return lambda feature: encoder(index(feature))

## Layer 3: Normalization Layer

In [ ]:
def create_normalization_layer(
    name: str, dataset: tf.data.Dataset
) -> tf.keras.layers.Layer:
    """
    Create a normalization layer for a numerical feature.
    Args:
        name (str): The feature name in the `features` dict.
        dataset (tf.data.Dataset): The dataset yielding (features, label,
        sample_weight). Each `features` is a dict that must contain `name`.
    Returns:
        tf.keras.layers.Layer: A `Normalization` layer.
    """
    normalizer = tf.keras.layers.Normalization(
        axis=None,  # or axis=-1 if each feature is a 1D vector to normalize per component
        name=name + "_normalizer",
    )
    feature_ds = dataset.map(
        lambda features, label, sample_weight: features[name],
        name=f"{name}_normalizer_map",
    ).unbatch()  # Flatten the dataset to iterate over all values

    normalizer.adapt(feature_ds)

    return normalizer

## Layer 4: Dynamic Encoding Layer (`Layer + Layer Creation Helper`)

In [ ]:
class DynamicCategoryEncoding(tf.keras.layers.Layer):
    """
    Custom layer to encode string categories to integer codes using a precomputed vocabulary.
    """

    def __init__(
        self,
        action_space_map: dict,
        name: str = "dynamic_category_encoding",
        oov_value: int = None,
        **kwargs,
    ):
        """
        Initialize a DynamicCategoryEncoding layer with a dictionary from `calc_action_space`.
        Args:
            action_space_map (dict): Must contain "unique_actions" and
            "action_counts". Typically, this is the output of calc_action_space(dataset).
            name (str): Name for the layer.
            oov_value (int): The integer to assign to out-of-vocabulary items (default: -99).
        """
        super(DynamicCategoryEncoding, self).__init__(name=name, **kwargs)
        if (
            not action_space_map
            or "unique_actions" not in action_space_map
            or "action_counts" not in action_space_map
        ):
            raise ValueError(
                "Invalid or empty action_space_map provided. "
                "Verify 'unique_actions' and 'action_counts' keys exist."
            )

        self.action_data = action_space_map
        self.unique_actions = self.action_data["unique_actions"]
        self.action_counts = self.action_data["action_counts"]
        self.oov_value = oov_value or (len(self.unique_actions) + 1)

        self.action_mapping = tf.lookup.StaticVocabularyTable(
            initializer=tf.lookup.KeyValueTensorInitializer(
                keys=tf.constant(self.unique_actions, dtype=tf.string),
                values=tf.range(len(self.unique_actions), dtype=tf.int64),
            ),
            num_oov_buckets=self.oov_value,
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Encodes string inputs to integer codes.
        Args:
            inputs (tf.Tensor): A 1D or 2D string tensor (batch dimension optional).
        Returns:
            tf.Tensor: Encoded integer tensor. OOV items are forced to `self.oov_value`.
        Note:
            If the bucket is out-of-vocab, it will produce an ID >= len(self.unique_actions).
        """
        encoded = self.action_mapping.lookup(inputs)
        return tf.where(
            encoded < len(self.unique_actions),
            encoded,
            tf.constant(self.oov_value, dtype=encoded.dtype),
        )

    def get_category_mapping(self) -> dict:
        """
        Retrieve the category-to-integer mapping.
        """
        return {action: idx for idx, action in enumerate(self.unique_actions)}

    def get_integer_mapping(self) -> dict:
        """
        Retrieve the integer-to-category mapping.
        """
        return {idx: action for idx, action in enumerate(self.unique_actions)}


def create_dynamic_category_encoding_layer(
    dataset: tf.data.Dataset,
    feature_name: str = ACTION_WEIGHT_COLUMN,
    layer_name: str = "dynamic_category_encoding",
    oov_value: int = None,
    **kwargs,
) -> DynamicCategoryEncoding:
    """
    Factory function to create a DynamicCategoryEncoding layer.
    Args:
        dataset (tf.data.Dataset): Each item should be (features, label, sample_weight).
        feature_name (str): The key in `features` dict that you want to encode.
        layer_name (str): Name to give the DynamicCategoryEncoding layer.
        oov_value (int): Value to assign for out-of-vocabulary items in the final encoding.
    Returns:
        DynamicCategoryEncoding: A fully instantiated encoding layer ready for usage.
    """
    feature_ds = dataset.map(
        lambda features, label, sample_weight: {feature_name: features[feature_name]},
        name=f"{feature_name}_extraction_map",
    )
    action_space_map = calc_action_space(feature_ds)

    return DynamicCategoryEncoding(
        action_space_map=action_space_map,
        name=layer_name,
        oov_value=oov_value,
        **kwargs,
    )

****Required Inputs**:

- **states**: all columns except 'presented_powerup', 'is_powerup_clicked'
- **actions**: 'presented_powerup' as category codes
- **weights**: weighted_rewards_layer using as inputs 'presented_powerup' (actions) and 
'is_powerup_clicked' (rewards)

# Single Model With Preprocessing + Q-Network

### 1. Preprocessing Layers: Prebuild and “adapt” the layers

Because Normalization and StringLookup (or your custom DynamicCategoryEncoding) must scan the data once to learn vocab/stats, we can do something like this outside the model class

In [ ]:
def create_preprocessing_submodel(
    input_cols: dict,
    action_col: str,
    dataset: tf.data.Dataset,
) -> tf.keras.Model:
    """
    Build a functional sub-model by creating and adapting the preprocessing layers.
    Args:
        input_cols (dict): A dictionary mapping feature names to data types.
        action_col (str): The column name for the action.
        dataset (tf.data.Dataset): The dataset to adapt the layers.
    Returns:
        tf.keras.Model: A functional sub-model for preprocessing.
    """
    logging.info("1 - Creating layers")

    logging.info("Taking raw columns as separate Input(...) layers")
    inputs_dict = {}
    for colname, colinfo in input_cols.items():
        inputs_dict[colname] = Input(name=colname, shape=(1,), dtype=colinfo["type"])

    logging.info("Normalization or One-hot")
    numeric_layers = {}
    string_layers = {}
    for colname, colinfo in input_cols.items():
        if colname == action_col:
            continue
        if colinfo["type"] in [
            tf.float32,
            tf.int32,
            tf.int64,
        ]: # numeric => normalization
            numeric_layers[colname] = create_normalization_layer(colname, dataset)
        elif colinfo["type"] == tf.string:  # string => one-hot
            string_layers[colname] = create_one_hot_encoding_layer(
                name=colname,
                dataset=dataset,
                dtype="string",
            )

    logging.info("Encoding Actions into Integer IDs")
    action_encoder = create_dynamic_category_encoding_layer(
        dataset=dataset,
        feature_name=action_col,
        layer_name=f"{action_col}_encoder",
    )

    logging.info("2 - Building & adapting functional Graph for preprocessing")
    transformed_tensors = []
    for colname, inp in inputs_dict.items():
        if colname == action_col:
            continue
        fillna_layer = FillNA(name=f"fill_na_{colname}")
        filled = fillna_layer(inp)

        colinfo = input_cols[colname]
        if colinfo["type"] in [tf.float32, tf.int32, tf.int64]:
            norm = numeric_layers[colname](filled)  # shape: (batch, ) if axis=None
            reshape = Reshape((1,))  # reshape so we can concat with one-hot
            reshape.name = f"{colname}_reshape"
            norm = reshape(norm)
            transformed_tensors.append(norm)
        else:
            # string => one-hot
            oh = string_layers[colname](filled)  # shape: (batch, X)
            transformed_tensors.append(oh)

    logging.info("Output 1: Concatenating all non-action features")
    if len(transformed_tensors) > 1:
        concat_features = Concatenate(axis=-1, name="concat_all")(transformed_tensors)
    else:
        concat_features = transformed_tensors[0]

    logging.info("Output 2: Encoding the actions feature")
    action_inp = inputs_dict[action_col]
    fillna_layer = FillNA(name=f"fill_na_{colname}")
    action_filled = fillna_layer(action_inp)
    action_id = action_encoder(action_filled)  # shape: (batch, ) or (batch, 1)

    logging.info("3 - Building the sub-model")
    submodel = Model(
        inputs=list(inputs_dict.values()),
        outputs=[concat_features, action_id],
        name="preprocessing_submodel",
    )

    action_mapping = action_encoder.get_integer_mapping()
    logging.info(f"Saving Action Mapping: {action_mapping}")
    submodel.action_mapping = action_mapping

    return submodel

Generally, we’ll **adapt on the "training" portion of your data ONLY**. Here’s why:

- **Avoid data leakage**: If you include validation or test data when learning normalization parameters (mean, variance) or building a vocabulary, you’re leaking future (unseen) information into your training pipeline. This can inflate performance metrics artificially.
- **Real-world training**: Typically, in production you only have your “training data” to estimate distribution statistics. Validation/test sets are specifically held out to measure performance on new, unseen data.

In [ ]:
logging.info("Adapting the preprocessing layers to the dataset...")
preproc_model = create_preprocessing_submodel(
    input_cols=INPUT_COLS,
    action_col=ACTION_WEIGHT_COLUMN,
    dataset=train_dataset,
)

In [ ]:
print(preproc_model.action_mapping)

In [ ]:
tf.keras.utils.plot_model(
    preproc_model,
    show_layer_names=True,
    expand_nested=True,
    show_trainable=True,
    rankdir="LR",
    show_layer_activations=True,
    show_shapes=True,
    to_file=KERAS_DIR + "preprocessing_model.png",
)

### 2. Full Bandit Model: Build a single Model with those layers inside __init__

Now we create a custom model that:

Has a `call(...)` method that does:

1. The pre-adapted data processing (raw features -> fillNA -> normalization/one-hot -> concat).
2. Compute the Q-values using a Dense Neural Network (Q-network).

Then a custom train_step/test_step for the bandit logic.

In [ ]:
class NeuralBanditModel(tf.keras.Model):
    def __init__(
        self, preprocessing_submodel: tf.keras.Model, output_dim: int, **kwargs
    ):
        """
        Initialize and build a "Multi-Layer Perceptron" using Preprocessing designed
        as a Functional API, which is adapted to the dataset before building the
        NeuralBanditModel model.
        The network is designed to learn Q-values for a bandit problem, where
        `output_dim` corresponds to the number of distinct actions (e.g., powerups).
        Args:
          preprocessing_submodel: The functional model that outputs (concat_features,
           action_id).
          output_dim: Number of possible actions => dimension of Q-values.
        """
        super().__init__(**kwargs)
        self.preproc_model = preprocessing_submodel
        self.output_dim = output_dim

        self.qnet = tf.keras.Sequential(  # Build a Dense Neural Q-network
            [
                tf.keras.layers.Dense(256, activation="relu", name="hidden_dense_1"),
                tf.keras.layers.Dense(512, activation="relu", name="hidden_dense_2"),
                tf.keras.layers.Dense(512, activation="relu", name="hidden_dense_3"),
                tf.keras.layers.Dense(256, activation="relu", name="hidden_dense_4"),
                tf.keras.layers.Dropout(0.2, name="dropout_1"),
                tf.keras.layers.Dense(128, activation="relu", name="hidden_dense_5"),
                tf.keras.layers.Dense(64, activation="relu", name="hidden_dense_6"),
                tf.keras.layers.Dropout(0.2, name="dropout_2"),
                tf.keras.layers.Dense(32, activation="relu", name="hidden_dense_7"),
                tf.keras.layers.Dense(
                    output_dim, activation="relu", name="output_dense"
                ),  # Q-values output: one per action ("powerup")
            ],
            name="neural_bandit_q_network",
        )

    def call(self, inputs: dict, training: bool = False):
        """
        Forward pass:
         1) Run raw inputs through the preprocessing sub-model, pre-adapted to the
         dataset.
         2) Q-network forward pass on `concat_features`.
         3) Return (q_values, action_id) so we can build bandit logic in prediction
         within the `train_step`.
         Args:
            inputs: A dictionary of raw features
            training (bool): If True, layers like Dropout & Batch Normalization will
            work in training mode; otherwise, they're in inference mode.
        Note:
            Batch Normalization is a pre-processing layers, so we don't need to
            consider the training mode here (pre-adapted).
        """
        concat_features, action_id = self.preproc_model(inputs, training=training)
        q_values = self.qnet(concat_features, training=training)
        return q_values, action_id

    def train_step(self, data: tuple):
        """
        Custom training logic for the bandit approach. Keras calls this method once
        per batch during model.fit().
        Note: the one-hot mask is applied to the Q-values to select the predicted,
        this to compute the loss only for the chosen action.
        Args:
            data (tuple):
                (features, label, sample_weight).
                - features is a dict with multiple columns (distance_avg, coins_spent,
                  geo_country, etc.).
                - label (rewards) is a float tensor (is_powerup_clicked)
                representing the reward for the chosen action.
                - sample_weight (action weights e.g.: 1.0, 4.12, 1.0, 4.34, ...) is 
                an optional tensor or None for weighting the loss function.
        """
        features, label, sample_weight = data

        with tf.GradientTape() as tape:
            # Forward pass: get Q-values and integer action_id
            q_values, action_id = self(features, training=True)  # True: training mode

            # Bandit logic: build one-hot mask
            action_id = tf.reshape(action_id, [-1])  # shape=(batch,)
            action_mask = tf.one_hot(
                tf.cast(action_id, tf.int32), depth=self.output_dim
            )

            # Overwrite predicted Q-values with the actual rewards
            label = tf.reshape(label, [-1, 1])  # shape=(batch,1)
            target = q_values * (1.0 - action_mask) + label * action_mask

            loss = self.compute_loss(  # Compute sample-weighted loss
                features,  # x
                target,    # y
                q_values,  # y_pred
                sample_weight=sample_weight,
                training=True,
            )

        grads = tape.gradient(
            loss, self.trainable_variables
        )  # Apply gradients (backprop)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        # Update metrics - sample_weight is used to compute the weighted loss
        self.compute_metrics(features, target, q_values, sample_weight=sample_weight)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data: tuple):
        """
        Custom evaluation logic, similar to train_step but no gradient updates.
        """
        features, label, sample_weight = data
        q_values, action_id = self(features, training=False)  # False: inference mode

        action_id = tf.reshape(action_id, [-1])
        action_mask = tf.one_hot(tf.cast(action_id, tf.int32), depth=self.output_dim)

        label = tf.reshape(label, [-1, 1])
        target = q_values * (1.0 - action_mask) + label * action_mask

        loss = self.compute_loss(
            features, target, q_values, sample_weight=sample_weight, training=False
        )
        self.compute_metrics(features, target, q_values, sample_weight=sample_weight)

        return {m.name: m.result() for m in self.metrics}

    def get_config(self):
        """
        Placing serializable items, excluding preproc_model because it's too complex
        and non-serializable
        """
        config = super().get_config()
        config.update(
            {
                "output_dim": self.output_dim,
            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        """
        Create an instance with a placeholder for the preprocessing submodel, must
        reattach the actual preprocessing submodel after loading.
        """
        output_dim = config.pop("output_dim")
        return cls(preprocessing_submodel=None, output_dim=output_dim, **config)

## Custom callback to evaluate model

After each epoch, we will test our model against validation data to see how it's performing relative to the benchmark. We will use random selection of actions as the benchmark.

To test our data, we will first **filter for the samples that yield a positive reward**, and **see if our model can predict the action that generated that positive reward**.

Because our dataset might be biased (one action appearing more frequently than others), we need to **downsample all actions so they are evenly distributed**, otherwise the test result will be biased. For example, if `action_1` is appears twice as frequently as the other actions, a model that only predicts `action_1` will be "twice as good as the benchmark", where in reality this will not be true.

In [ ]:
class ValidationCallback(tf.keras.callbacks.Callback):
    """
    Custom callback that, at the end of each epoch, evaluates the model on the
    evaluation dataset by filtering for samples with a positive reward
    (is_powerup_clicked == 1) and then checks if the model would choose the same
    action (powerup) that generated that reward.
        Note: To avoid bias (if one action is over-represented), the results are
        computed on a balanced subset where each action appears the same number of
        times.
    """

    def __init__(self, eval_dataset, **kwargs):
        """
        Args:
            eval_dataset (tf.data.Dataset): A tf.Dataset that yields tuples
              (features, label, sample_weight). It is assumed that label==1 indicates
               a positive reward.
        """
        super().__init__(**kwargs)
        self.eval_dataset = eval_dataset

    def on_epoch_end(self, epoch, logs=None):
        """
        At the end of each epoch, evaluate the model on the evaluation dataset.
        Steps performed:
            1. Filter for samples with a positive reward.
            2. Downsample each action class to the same sample size.
            3. Predict on the resulting subset.
            4. Check how often the model predicts the actual action that yielded reward = 1.
        """
        logs = logs or {}
        all_pred = []
        all_true = []

        for batch in self.eval_dataset:
            features, label, sample_weight = batch
            label = tf.reshape(label, [-1])
            pos_mask = tf.equal(label, 1.0)  # mask to keep only positive rewards
            if not tf.reduce_any(pos_mask):
                continue

            pos_indices = tf.where(pos_mask)[:, 0]
            pos_features = {
                key: tf.gather(val, pos_indices) for key, val in features.items()
            }

            # Inference on the positive samples, where true_action is the ground truth
            q_values, true_action = self.model(pos_features, training=False)

            pred_action = tf.argmax(q_values, axis=-1, output_type=tf.int32)
            true_action = tf.reshape(true_action, [-1])

            all_pred.append(pred_action)
            all_true.append(true_action)

        if not all_pred:
            print(
                f"Epoch {epoch+1}: No positive reward samples found in evaluation dataset."
            )
            return  # exit >>

        # Concatenate predictions and ground truth from all batches.
        all_pred = tf.concat(all_pred, axis=0)
        all_true = tf.concat(all_true, axis=0)

        # Downsample: to avoid bias from unequal action frequencies
        unique_actions, _ = tf.unique(
            all_true
        )  # find the unique action IDs in the positive reward set
        unique_actions_np = unique_actions.numpy()

        counts = {}
        # Counts the number of samples for each action class
        for action in unique_actions_np:
            mask = tf.equal(all_true, tf.cast(action, all_true.dtype))
            counts[action] = int(tf.reduce_sum(tf.cast(mask, tf.int32)).numpy())

        if len(counts) == 0:
            print(
                f"Epoch {epoch+1}: No positive reward samples available after filtering."
            )
            return

        min_count = min(counts.values())
        if min_count == 0:
            print(
                f"Epoch {epoch+1}: One or more action classes have zero samples. "
                f"Skipping balanced evaluation."
            )
            return

        # For each action, randomly shuffle the indices and pick min_count examples
        balanced_indices = []
        for action in unique_actions_np:
            mask = tf.equal(all_true, tf.cast(action, all_true.dtype))
            indices = tf.where(mask)[:, 0]
            shuffled_indices = tf.random.shuffle(indices)
            balanced_indices.append(shuffled_indices[:min_count])

        # Measure the model's accuracy on the balanced subset
        balanced_indices = tf.concat(balanced_indices, axis=0)
        balanced_indices = tf.sort(balanced_indices)

        balanced_pred = tf.gather(all_pred, balanced_indices)
        balanced_true = tf.gather(all_true, balanced_indices)

        balanced_true = tf.cast(balanced_true, tf.int32)
        correct = tf.cast(tf.equal(balanced_pred, balanced_true), tf.float32)
        # fraction of samples where the predicted action matches the ground truth
        balanced_accuracy = tf.reduce_mean(correct)

        print(
            f"Epoch {epoch+1}: Balanced Accuracy on Positive Reward Samples: {balanced_accuracy:.4f}"
        )
        logs["action_accuracy"] = balanced_accuracy.numpy()

# Compile the Model

In [ ]:
action_space = calc_action_space(
    dataset.map(lambda f, l, sw: {ACTION_WEIGHT_COLUMN: f[ACTION_WEIGHT_COLUMN]})
)
OUTPUT_DIM = action_space["action_space_size"]
print("Total Actions:", OUTPUT_DIM)

bandit_model = NeuralBanditModel(
    preprocessing_submodel=preproc_model,
    output_dim=OUTPUT_DIM,
)

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    INITIAL_LEARNING_RATE,  # Initial learning rate
    decay_steps=LR_DECAY_STEPS,  # Number of steps before applying the decay
    decay_rate=LR_DECAY_RATE,  # Factor by which the learning rate is decayed
    staircase=LR_STAIRCASE,  # If True, the decay occurs in discrete steps
)

bandit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[tf.keras.metrics.MeanSquaredError()],
    run_eagerly=False,  # Set to True for debugging; remove or set to False for production
)

# Train the Model

In [ ]:
state_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=KERAS_DIR + "NeuralBanditModel_state_checkpoint.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min",
)

bandit_model.fit(
    train_dataset.repeat(),
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=eval_dataset,
    callbacks=[ValidationCallback(eval_dataset), state_checkpoint],
)

# Evaluate the Model

In [ ]:
results = bandit_model.evaluate(eval_dataset)
print("Evaluation results:", results)

# Save the Model

In [ ]:
if SAVE_FORMAT in ["tf", "h5"]:
    if SAVE_FORMAT == "tf":
        bandit_model.save(KERAS_DIR + "NeuralBanditModel.keras")
    elif SAVE_FORMAT == "h5":
        bandit_model.save(H5_DIR + "NeuralBanditModel.h5", save_format="h5")
else:
    raise ValueError(f"Invalid save format: {SAVE_FORMAT}")

# Save Model to TFLite

In [ ]:
def export_tflite_model(
    model: NeuralBanditModel,
    input_cols: dict,
    train_dataset: tf.data.Dataset,
    output_tflite_path="NeuralBanditModel.tflite",
    preprocess_json_path="preprocess.json",
    metadata_filename="model_metadata.tflite",
    tflite_model_version: str = "v1",
):
    """
    Convert a trained TensorFlow model to TFLite, generate preprocessing metadata
    (using stats computed from train_dataset and the submodel.action_mapping), and
    attach the metadata to the TFLite model.
    Args:
        model: The trained model (NeuralBanditModel only).
        input_cols: A dictionary of input column definitions.
        train_dataset: A tf.data.Dataset used for training.
        output_tflite_path: Path to save the converted TFLite model.
        preprocess_json_path: Path to save the preprocessing metadata JSON.
        metadata_filename: Path to save the serialized metadata buffer (optional).
    """
    logging.info("1.1 - Extracting sample features for conversion")
    sample_batch = next(iter(train_dataset))
    sample_features = sample_batch[0]  # first batch is used as a representative sample
    sample_input = {
        key: value[:1] for key, value in sample_features.items()
    }  # 1 dim reduction

    logging.info("1.2 - Converting model to TFLite")
    concrete_func = tf.function(model).get_concrete_function(sample_input)
    converter = tf.lite.TFLiteConverter.from_concrete_functions(
        [concrete_func], trackable_obj=model
    )
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]  # !!! Enabling TensorFlow Select ops for any ops not supported natively
    tflite_model = converter.convert()

    with open(output_tflite_path, "wb") as f:
        f.write(tflite_model)
    logging.info(
        f"1.3 - Converted TF model to TFLite and saved at {output_tflite_path}"
    )

    logging.info("2.1 - Creating preprocessing metadata JSON")
    num_columns = [
        col
        for col, info in input_cols.items()
        if info["type"] in [tf.float32, tf.int32, tf.int64]
        and col != ACTION_WEIGHT_COLUMN
    ]
    cat_columns = [
        col
        for col, info in input_cols.items()
        if info["type"] == tf.string and col != ACTION_WEIGHT_COLUMN
    ]

    logging.info(
        "2.2 Extracting numerical feature stats from the preprocessing submodel"
    )
    numerical_stats = {}
    for col in num_columns:
        # Try getting Normalization layer from the preprocessing submodel (PRE-COMPUTED, its mean and variance after adapt)
        try:
            norm_layer = model.preproc_model.get_layer(f"{col}_normalizer")
            mean = (
                norm_layer.mean.numpy().tolist() if hasattr(norm_layer, "mean") else 0.0
            )
            variance = (
                norm_layer.variance.numpy().tolist()
                if hasattr(norm_layer, "variance")
                else 0.0
            )
            std = (variance**0.5) if variance else 0.0
            numerical_stats[col] = {"type": "numerical", "mean": mean, "std": std}
        except ValueError:
            # If the layer is not found, fall back to dummy values
            numerical_stats[col] = {"type": "numerical", "mean": 0.0, "std": 0.0}

    logging.info(
        "2.3 - Extracting categorical feature stats from the preprocessing submodel"
    )
    categorical_info = {}
    for col in cat_columns:
        # Try getting vocabulary from lookup layers
        try:
            lookup_layer = model.preproc_model.get_layer(f"{col}_string_lookup")
            vocab = (
                lookup_layer.get_vocabulary()
                if hasattr(lookup_layer, "get_vocabulary")
                else []
            )
            # Ensure all vocabulary entries are strings in UTF-8 format
            vocab = [v.decode("utf-8") if isinstance(v, bytes) else v for v in vocab]
        except ValueError:
            vocab = ["dummy"]
        categorical_info[col] = {"type": "categorical", "all_values": vocab}

    logging.info("2.4 - Extracting action mapping from the preprocessing submodel")
    actions_mapping = model.preproc_model.action_mapping
    output_mapping = [
        val.decode("utf-8") if isinstance(val, bytes) else val
        for val in actions_mapping.values()
    ]

    logging.info("2.5 - Saving preprocessing metadata to JSON")
    preprocess_info = {}
    preprocess_info.update(numerical_stats)
    preprocess_info.update(categorical_info)
    preprocess_info["output_mapping"] = output_mapping

    with open(preprocess_json_path, "w") as f:
        json.dump(preprocess_info, f, indent=4)
    print(f"Preprocessing metadata saved to {preprocess_json_path}")
    logging.info("2.6 - Preprocessing metadata saved to {preprocess_json_path}")

    logging.info("3.1 - Attaching metadata to the TFLite model")
    model_meta = _metadata_fb.ModelMetadataT()
    model_meta.name = "NeuralBanditModel"
    model_meta.description = (
        "This model outputs Q-values for each powerup action given a user's state. "
        "It uses a preprocessing submodel with normalization for numerical features and "
        "lookup-based encoding for categorical features. The attached JSON file contains "
        "the dynamically computed preprocessing parameters."
    )
    model_meta.version = tflite_model_version

    logging.info("3.2 - Serializing metadata to a buffer")
    builder = flatbuffers.Builder(0)
    builder.Finish(
        model_meta.Pack(builder), _metadata.MetadataPopulator.METADATA_FILE_IDENTIFIER
    )
    metadata_buf = builder.Output()

    logging.info(f"3.3 - Saving metadata to {metadata_filename}")
    with open(metadata_filename, "wb") as f:
        f.write(metadata_buf)

    logging.info("3.4 - Attaching metadata to the TFLite model")
    populator = _metadata.MetadataPopulator.with_model_file(output_tflite_path)
    populator.load_associated_files([preprocess_json_path])
    populator.populate()

    logging.info("--- Metadata successfully attached to the TFLite model ---")


export_tflite_model(
    model=bandit_model,
    input_cols=INPUT_COLS,
    train_dataset=train_dataset,
    output_tflite_path=TFLITE_DIR + "NeuralBanditModel.tflite",
    preprocess_json_path=TFLITE_DIR + "preprocess.json",
    metadata_filename=TFLITE_DIR + "model_metadata.tflite",
    tflite_model_version="v1",
)

# Load the Model

In [ ]:
# Load the model (the custom_objects argument ensures your custom class is used)
loaded_model = tf.keras.models.load_model(
    KERAS_DIR + "NeuralBanditModel.keras",
    custom_objects={"NeuralBanditModel": NeuralBanditModel},
)
# Reattach the preprocessing submodel (which you have already created/adapted)
loaded_model.preproc_model = preproc_model

# Make Predictions

In [ ]:
for features, labels, sample_weights in eval_dataset.take(1):
    q_values, action_ids = bandit_model(features)

    q_values_np = q_values.numpy()
    action_ids_np = action_ids.numpy().flatten()

    for q_val, act_id in zip(q_values_np, action_ids_np):
        # If you want the mapped action (assuming action_mapping is a dict with integer keys)
        action_label = preproc_model.action_mapping[act_id]
        print(
            f"q_values: {q_val} | action_id: {act_id} | mapped action: {action_label} \n"
        )

# Display the model architecture

In [ ]:
bandit_model.summary(
    line_length=112,
    positions=None,
    expand_nested=True,
    print_fn=lambda x: print(x, file=open(KERAS_DIR + "model_summary.txt", "a")),
)

In [ ]:
tf.keras.utils.plot_model(
    bandit_model.qnet,
    show_layer_names=True,
    expand_nested=True,
    show_trainable=True,
    rankdir="LR",
    show_layer_activations=True,
    show_shapes=True,
    to_file=KERAS_DIR + "bandit_model.png",
)

# Considerations (ToDos)

- preprocessing layer must be recomputed after loading the model, can not be 
serialized (find a way to save the functional preprocessing models)
- TFLite is experimental, have never been tested
- Add timestamps to models

FileNotFoundError: Config file not found: config/default_config.yaml